# Qwen2-VL Multi-Hypothesis Grounding + Blind Verification

This notebook compares:

- **A. Structured baseline**: existing LGT order-refine adapter, pairwise/first/last structured decoding.
- **C. Multi-hypothesis grounding + blind verification**: caption event hypotheses and image evidence are generated independently, all 24 permutations are jointly scored, and only the disputed top-1/top-2 relation is verified blindly.

The notebook intentionally excludes single-path multi-turn memory and top-5 round-robin reranking. It runs quick50, tuning150, and holdout150 before any train split cache or fine-tuning data generation.


In [ ]:
# 1) Install dependencies, then restart runtime once.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_multi_hypothesis_refine_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")


In [ ]:
# 2) Setup
from google.colab import drive
drive.mount("/content/drive")

import ast
import gc
import glob
import hashlib
import itertools
import json
import math
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
import random
import re
import shutil
import time
import zipfile
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = None
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None
try:
    from transformers import AutoModelForVision2Seq
except ImportError:
    AutoModelForVision2Seq = None
from peft import PeftModel, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")
warnings.filterwarnings("ignore", message=".*The following generation flags are not valid.*")
transformers_logging.set_verbosity_error()

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

MODEL_REPO_ID = "Qwen/Qwen2-VL-2B-Instruct"
USE_MODELSCOPE_BASE_MODEL = True
DRIVE_MODEL_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/model_cache/Qwen2-VL-2B-Instruct"
INITIAL_ADAPTER_DIR = (
    "/content/drive/MyDrive/SNU_AI_Challenge/"
    "qwen2vl_lgt_order_refine_v1/runs/20260714_003635/"
    "lgt_order_refine/best_adapter"
)
SPLIT_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/id_splits/qwen2vl_lgt_order_refine_20260714_003635"
OUTPUT_ROOT = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_multi_hypothesis_refine_v1"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = os.path.join(OUTPUT_ROOT, "runs", RUN_ID)
OUTPUT_DIR = os.path.join(RUN_ROOT, "multi_hypothesis_refine")
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
CACHE_ROOT = os.path.join(OUTPUT_ROOT, "shared_cache")
BASELINE_CACHE_DIR = os.path.join(CACHE_ROOT, "baseline_structured_cache")
TEXT_CACHE_DIR = os.path.join(CACHE_ROOT, "text_hypothesis_cache")
VISUAL_CACHE_DIR = os.path.join(CACHE_ROOT, "visual_evidence_cache")
VERIFY_CACHE_DIR = os.path.join(CACHE_ROOT, "focused_verification_cache")
RUN_SPLIT_DIR = os.path.join(RUN_ROOT, "splits")

SEED = 42
VALID_RATIO = 0.10
QUICK_EVAL_ROWS = 50
TUNING_EVAL_ROWS = 150
HOLDOUT_EVAL_ROWS = 150

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

STRUCTURED_ALPHA = 1.0
STRUCTURED_BETA = 1.0
STRUCTURED_GAMMA = 1.0
PAIRWISE_BIDIRECTIONAL = False
BASELINE_ROW_BATCH_SIZE = 2
TEXT_BATCH_SIZE = 8
VISUAL_BATCH_SIZE = 2
K_TEXT_HYPOTHESES = 2
MAX_NEW_TOKENS_TEXT = 384
MAX_NEW_TOKENS_VISUAL = 256
MAX_NEW_TOKENS_VERIFY = 16
PROMPT_VERSION = "multi_hypothesis_event_grounding_v2"

# Pilot C scoring: keep transition/temporal at zero so improvements are attributable
# to independent visual grounding plus the original structured decoder score.
WEIGHT_PRESETS = [
    {"name": "structured_grounding_070", "structured": 0.70, "grounding": 0.30, "transition": 0.0, "temporal": 0.0},
    {"name": "grounding_050", "structured": 0.50, "grounding": 0.50, "transition": 0.0, "temporal": 0.0},
]
UNCERTAIN_QUANTILE_GRID = [0.20, 0.30, 0.40]
VERIFY_MARGIN_GRID = [0.10]
SWITCH_MARGIN_GRID = [0.15, 0.20]
ORDER_SWITCH_GAIN_GRID = [0.05, 0.10]
KENDALL_SWITCH_MAX = 1
MAX_VERIFICATIONS_GRID = [1]
FORWARD_REVERSE_AGREEMENT_MIN = 0.80

TRAIN_ROWS_FOR_RECORDS = 1000
TRAIN_PILOT_MAX_STEPS = 500
TRAIN_PILOT_SAVE_STEPS = 100
TRAIN_PILOT_LEARNING_RATE = 1e-6
TRAIN_PILOT_BATCH_SIZE = 1
TRAIN_PILOT_GRAD_ACCUM = 4
CATEGORY_RATIOS = {
    "recovery": 0.35,
    "hard_preservation": 0.35,
    "easy_preservation": 0.15,
    "ambiguous": 0.15,
}
RECOVERY_RATIO = CATEGORY_RATIOS["recovery"]
PRESERVATION_RATIO = CATEGORY_RATIOS["hard_preservation"] + CATEGORY_RATIOS["easy_preservation"]
AMBIGUOUS_RATIO = CATEGORY_RATIOS["ambiguous"]

for path in [OUTPUT_ROOT, RUN_ROOT, OUTPUT_DIR, EVAL_DIR, CACHE_ROOT, BASELINE_CACHE_DIR, TEXT_CACHE_DIR, VISUAL_CACHE_DIR, VERIFY_CACHE_DIR, RUN_SPLIT_DIR, SPLIT_DIR]:
    os.makedirs(path, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.exists(os.path.join(INITIAL_ADAPTER_DIR, "adapter_config.json")), INITIAL_ADAPTER_DIR

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("output:", OUTPUT_DIR)
print("adapter:", INITIAL_ADAPTER_DIR)


In [ ]:
# 3) IO, split, metrics, and cache helpers
def save_json(data, path):
    os.makedirs(os.path.dirname(str(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_jsonl(records, path):
    os.makedirs(os.path.dirname(str(path)), exist_ok=True)
    partial = str(path) + ".partial"
    with open(partial, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    os.replace(partial, path)


def append_jsonl(record, path):
    os.makedirs(os.path.dirname(str(path)), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def sha16(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:16]


def sha16_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()[:16]


def adapter_fingerprint(adapter_dir):
    files = []
    for name in ["adapter_model.safetensors", "adapter_model.bin", "pytorch_model.bin"]:
        candidate = os.path.join(adapter_dir, name)
        if os.path.exists(candidate):
            stat = os.stat(candidate)
            files.append({"name": name, "size": int(stat.st_size), "mtime_ns": int(stat.st_mtime_ns), "sha16": sha16_file(candidate)})
    return {
        "adapter_dir": adapter_dir,
        "adapter_config_hash": sha16(load_json(os.path.join(adapter_dir, "adapter_config.json"))),
        "weight_files": files,
    }


def ids_hash(ids):
    return sha16("\n".join(str(value) for value in ids))


def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def sequence_to_answer(order):
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


def row_image_paths(row, image_root):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def rows_by_ids(dataframe, ids):
    ids = [str(value) for value in ids]
    subset = dataframe[dataframe["Id"].isin(ids)].copy()
    order = {sample_id: index for index, sample_id in enumerate(ids)}
    subset["_split_order"] = subset["Id"].map(order)
    return subset.sort_values("_split_order").drop(columns=["_split_order"]).reset_index(drop=True)


def make_or_load_split_ids(dataframe):
    names = ["train_ids.json", "validation_ids.json", "quick50_ids.json", "tuning150_ids.json", "holdout150_ids.json"]
    paths = {name: os.path.join(SPLIT_DIR, name) for name in names}
    split_ids = {}
    if os.path.exists(paths["train_ids.json"]) and os.path.exists(paths["validation_ids.json"]):
        split_ids["train_ids.json"] = [str(value) for value in load_json(paths["train_ids.json"])]
        split_ids["validation_ids.json"] = [str(value) for value in load_json(paths["validation_ids.json"])]
    else:
        unique_ids = dataframe["Id"].unique().astype(str).copy()
        rng = np.random.default_rng(SEED)
        rng.shuffle(unique_ids)
        valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
        split_ids["validation_ids.json"] = list(unique_ids[:valid_size])
        split_ids["train_ids.json"] = list(unique_ids[valid_size:])
        save_json(split_ids["train_ids.json"], paths["train_ids.json"])
        save_json(split_ids["validation_ids.json"], paths["validation_ids.json"])

    validation_ids = list(split_ids["validation_ids.json"])
    rng = np.random.default_rng(SEED)
    shuffled = validation_ids.copy()
    rng.shuffle(shuffled)
    derived = {
        "quick50_ids.json": shuffled[:min(QUICK_EVAL_ROWS, len(shuffled))],
        "tuning150_ids.json": shuffled[min(QUICK_EVAL_ROWS, len(shuffled)):min(QUICK_EVAL_ROWS + TUNING_EVAL_ROWS, len(shuffled))],
        "holdout150_ids.json": shuffled[min(QUICK_EVAL_ROWS + TUNING_EVAL_ROWS, len(shuffled)):min(QUICK_EVAL_ROWS + TUNING_EVAL_ROWS + HOLDOUT_EVAL_ROWS, len(shuffled))],
    }
    for name, ids in derived.items():
        if os.path.exists(paths[name]):
            split_ids[name] = [str(value) for value in load_json(paths[name])]
        else:
            split_ids[name] = ids
            save_json(ids, paths[name])
    for name, ids in split_ids.items():
        save_json(ids, os.path.join(RUN_SPLIT_DIR, name))
    return split_ids


def order_metric_row(pred_order, gold_order):
    pred_order = [int(x) for x in pred_order]
    gold_order = [int(x) for x in gold_order]
    pred_ranks = {image_number: position for position, image_number in enumerate(pred_order)}
    gold_ranks = {image_number: position for position, image_number in enumerate(gold_order)}
    pair_accuracy = np.mean([
        (pred_ranks[a] < pred_ranks[b]) == (gold_ranks[a] < gold_ranks[b])
        for a, b in itertools.combinations([1, 2, 3, 4], 2)
    ])
    return {
        "exact_match": float(pred_order == gold_order),
        "pair_accuracy": float(pair_accuracy),
        "position_accuracy": float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
        "first_accuracy": float(pred_order[0] == gold_order[0]),
        "last_accuracy": float(pred_order[-1] == gold_order[-1]),
        "both_endpoint_accuracy": float(pred_order[0] == gold_order[0] and pred_order[-1] == gold_order[-1]),
    }


def safe_json_loads(text, fallback):
    if text is None:
        return fallback
    cleaned = str(text).strip()
    cleaned = re.sub(r"^```(?:json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    match = re.search(r"\{.*\}", cleaned, flags=re.S)
    if match:
        cleaned = match.group(0)
    try:
        return json.loads(cleaned)
    except Exception:
        return fallback


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)
train_df["gold_order"] = train_df["Answer_list"].apply(order_to_sequence)

split_ids = make_or_load_split_ids(train_df)
training_df = rows_by_ids(train_df, split_ids["train_ids.json"])
validation_df = rows_by_ids(train_df, split_ids["validation_ids.json"])
quick50_df = rows_by_ids(train_df, split_ids["quick50_ids.json"])
tuning150_df = rows_by_ids(train_df, split_ids["tuning150_ids.json"])
holdout150_df = rows_by_ids(train_df, split_ids["holdout150_ids.json"])

print("rows:", {"train": len(training_df), "validation": len(validation_df), "quick50": len(quick50_df), "tuning150": len(tuning150_df), "holdout150": len(holdout150_df)})
print("split hashes:", {name: ids_hash(ids) for name, ids in split_ids.items()})


In [ ]:
# 4) Model loading and prompt helpers
def model_cache_is_complete(model_dir):
    if not os.path.exists(os.path.join(model_dir, "config.json")):
        return False
    has_weight = any(os.path.exists(os.path.join(model_dir, name)) for name in ["model.safetensors.index.json", "pytorch_model.bin", "pytorch_model.bin.index.json"]) or bool(glob.glob(os.path.join(model_dir, "*.safetensors")))
    has_processor = any(os.path.exists(os.path.join(model_dir, name)) for name in ["preprocessor_config.json", "processor_config.json", "tokenizer.json", "tokenizer_config.json"])
    return bool(has_weight and has_processor)


def ensure_base_model_path():
    if model_cache_is_complete(DRIVE_MODEL_DIR):
        print("Using cached base model:", DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR
    if not USE_MODELSCOPE_BASE_MODEL:
        return MODEL_REPO_ID
    try:
        from modelscope import snapshot_download as modelscope_snapshot_download
        model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir="/content/modelscope_cache")
        os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)
        if not model_cache_is_complete(DRIVE_MODEL_DIR):
            tmp = DRIVE_MODEL_DIR + ".tmp"
            if os.path.exists(tmp):
                shutil.rmtree(tmp)
            shutil.copytree(model_dir, tmp, dirs_exist_ok=True)
            if os.path.exists(DRIVE_MODEL_DIR):
                shutil.rmtree(DRIVE_MODEL_DIR)
            os.replace(tmp, DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR
    except Exception as exc:
        print("ModelScope download failed; using HF repo id:", repr(exc))
        return MODEL_REPO_ID


def load_model_class():
    if Qwen2VLForConditionalGeneration is not None:
        return Qwen2VLForConditionalGeneration
    if AutoModelForImageTextToText is not None:
        return AutoModelForImageTextToText
    if AutoModelForVision2Seq is not None:
        return AutoModelForVision2Seq
    raise ImportError("No compatible Qwen2-VL model class found.")


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = os.path.isdir(MODEL_ID)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def model_device(active_model):
    return next(active_model.parameters()).device


def load_adapter_model(adapter_dir=INITIAL_ADAPTER_DIR):
    base = load_model_class().from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
        local_files_only=MODEL_LOCAL_FILES_ONLY,
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
    model.eval()
    model.config.use_cache = True
    if hasattr(model, "generation_config"):
        model.generation_config.do_sample = False
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None
        model.generation_config.num_beams = 1
    return model


def single_token_id(value):
    ids = processor.tokenizer.encode(str(value), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"{value!r} tokenized to {ids}; this notebook expects single-token labels.")
    return ids[0]


DIGIT_TOKEN_IDS = {digit: single_token_id(str(digit)) for digit in [1, 2, 3, 4]}


def chat_prompt(messages, add_generation_prompt=True):
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)


def user_message_with_images(image_count, instruction):
    content = []
    for idx in range(1, image_count + 1):
        content.append({"type": "text", "text": f"\nImage {idx}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + instruction})
    return [{"role": "user", "content": content}]


def user_message_text_only(instruction):
    return [{"role": "user", "content": [{"type": "text", "text": instruction}]}]


@torch.inference_mode()
def generate_text(active_model, messages, image_paths=None, max_new_tokens=256):
    image_paths = image_paths or []
    prompt = chat_prompt(messages, add_generation_prompt=True)
    old_padding_side = processor.tokenizer.padding_side
    try:
        processor.tokenizer.padding_side = "left"
        kwargs = {"text": [prompt], "return_tensors": "pt"}
        if image_paths:
            kwargs["images"] = [[load_rgb(path) for path in image_paths]]
        inputs = processor(**kwargs)
        inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
        generated = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
        new_tokens = generated[:, inputs["input_ids"].shape[1]:]
        return processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    finally:
        processor.tokenizer.padding_side = old_padding_side


print("digit token ids:", DIGIT_TOKEN_IDS)


In [ ]:
# 5) Structured baseline A: first/last/pairwise probabilities and 24-order scoring
def find_reference_run_config():
    candidates = [
        os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(INITIAL_ADAPTER_DIR))), "run_config.json"),
        os.path.join(os.path.dirname(os.path.dirname(INITIAL_ADAPTER_DIR)), "run_config.json"),
        os.path.join(os.path.dirname(INITIAL_ADAPTER_DIR), "run_config.json"),
        os.path.join(INITIAL_ADAPTER_DIR, "run_config.json"),
    ]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return None


def nested_get(mapping, path, default=None):
    current = mapping if isinstance(mapping, dict) else {}
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


def first_not_none(*values):
    for value in values:
        if value is not None:
            return value
    return None


INITIAL_BEST_CONFIG_PATH = os.path.join(INITIAL_ADAPTER_DIR, "best_config.json")
initial_best_config = load_json(INITIAL_BEST_CONFIG_PATH) if os.path.exists(INITIAL_BEST_CONFIG_PATH) else {}
reference_run_config_path = find_reference_run_config()
reference_run_config = load_json(reference_run_config_path) if reference_run_config_path else {}
reference_candidate_config = first_not_none(initial_best_config.get("candidate_generator"), reference_run_config.get("candidate_generator"), {})
reference_decoding_config = first_not_none(initial_best_config.get("decoding"), reference_run_config.get("decoding"), {})

STRUCTURED_ALPHA = float(first_not_none(
    nested_get(reference_candidate_config, ["alpha"]),
    nested_get(reference_decoding_config, ["alpha"]),
    initial_best_config.get("alpha"),
    reference_run_config.get("alpha"),
    STRUCTURED_ALPHA,
))
STRUCTURED_BETA = float(first_not_none(
    nested_get(reference_candidate_config, ["beta"]),
    nested_get(reference_decoding_config, ["beta"]),
    initial_best_config.get("beta"),
    reference_run_config.get("beta"),
    STRUCTURED_BETA,
))
STRUCTURED_GAMMA = float(first_not_none(
    nested_get(reference_candidate_config, ["gamma"]),
    nested_get(reference_decoding_config, ["gamma"]),
    initial_best_config.get("gamma"),
    reference_run_config.get("gamma"),
    STRUCTURED_GAMMA,
))
PAIRWISE_BIDIRECTIONAL = bool(first_not_none(
    nested_get(reference_candidate_config, ["pairwise_bidirectional"]),
    initial_best_config.get("pairwise_bidirectional"),
    reference_run_config.get("pairwise_bidirectional"),
    PAIRWISE_BIDIRECTIONAL,
))
MIN_PIXELS = int(first_not_none(nested_get(reference_candidate_config, ["min_pixels"]), reference_run_config.get("min_pixels"), MIN_PIXELS))
MAX_PIXELS = int(first_not_none(nested_get(reference_candidate_config, ["max_pixels"]), reference_run_config.get("max_pixels"), MAX_PIXELS))

reference_config_summary = {
    "best_config_path": INITIAL_BEST_CONFIG_PATH,
    "run_config_path": reference_run_config_path,
    "alpha": STRUCTURED_ALPHA,
    "beta": STRUCTURED_BETA,
    "gamma": STRUCTURED_GAMMA,
    "pairwise_bidirectional": PAIRWISE_BIDIRECTIONAL,
    "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS,
}
save_json(reference_config_summary, os.path.join(OUTPUT_DIR, "reference_config_summary.json"))
print("reference config summary:", json.dumps(reference_config_summary, ensure_ascii=False, indent=2))

# The processor in cell 4 may have been created before the reference run config
# overrode MIN_PIXELS/MAX_PIXELS. Recreate it here so image resolution matches
# the adapter configuration used by the structured baseline.
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
DIGIT_TOKEN_IDS = {digit: single_token_id(str(digit)) for digit in [1, 2, 3, 4]}
print("processor reloaded with reference resolution:", MIN_PIXELS, MAX_PIXELS)
print("digit token ids after reload:", DIGIT_TOKEN_IDS)
def original_task_instruction(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image occurs first?\n"
            "If the first image occurs earlier, answer 1.\n"
            "If the second image occurs earlier, answer 2.\n"
            "Answer only 1 or 2."
        )
    if task_type == "first":
        return f"Caption:\n{sentence}\n\nQuestion: Which image represents the beginning of the story?\nAnswer only the image number from 1 to 4."
    if task_type == "last":
        return f"Caption:\n{sentence}\n\nQuestion: Which image represents the end of the story?\nAnswer only the image number from 1 to 4."
    raise ValueError(task_type)


def make_original_eval_example(row, task_type, image_root, pair=None):
    image_paths = row_image_paths(row, image_root)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    return example


def original_messages(example):
    return user_message_with_images(len(example["image_paths"]), original_task_instruction(example))


@torch.inference_mode()
def score_digit_candidate_batch(active_model, examples, candidates_per_example):
    old_padding_side = processor.tokenizer.padding_side
    try:
        processor.tokenizer.padding_side = "right"
        texts = [chat_prompt(original_messages(example), add_generation_prompt=True) for example in examples]
        image_cache = {}
        def cached_load(path):
            if path not in image_cache:
                image_cache[path] = load_rgb(path)
            return image_cache[path]
        images = [[cached_load(path) for path in example["image_paths"]] for example in examples]
        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt")
        inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = active_model(**inputs)
        result = []
        for row_index, candidates in enumerate(candidates_per_example):
            last_pos = int(inputs["attention_mask"][row_index].sum().item()) - 1
            logits = outputs.logits[row_index, last_pos]
            token_ids = [DIGIT_TOKEN_IDS[int(candidate)] for candidate in candidates]
            probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
            result.append({int(candidate): float(prob) for candidate, prob in zip(candidates, probs)})
        return result
    finally:
        processor.tokenizer.padding_side = old_padding_side


def structured_score(first_probs, last_probs, pair_probs, order, alpha=STRUCTURED_ALPHA, beta=STRUCTURED_BETA, gamma=STRUCTURED_GAMMA):
    eps = 1e-12
    pair_score = np.mean([
        math.log(float(pair_probs[f"{order[i]}>{order[j]}"]) + eps)
        for i in range(4)
        for j in range(i + 1, 4)
    ])
    first_score = math.log(float(first_probs[str(order[0])]) + eps)
    last_score = math.log(float(last_probs[str(order[-1])]) + eps)
    return float(alpha * pair_score + beta * first_score + gamma * last_score)


def build_baseline_record(active_model, row, image_root, has_gold=True):
    examples = [make_original_eval_example(row, "first", image_root), make_original_eval_example(row, "last", image_root)]
    candidates = [[1, 2, 3, 4], [1, 2, 3, 4]]
    for first_index, second_index in PAIR_INDICES:
        a, b = first_index + 1, second_index + 1
        examples.append(make_original_eval_example(row, "pairwise", image_root, pair=(a, b)))
        candidates.append([1, 2])
        if PAIRWISE_BIDIRECTIONAL:
            examples.append(make_original_eval_example(row, "pairwise", image_root, pair=(b, a)))
            candidates.append([1, 2])
    probs = score_digit_candidate_batch(active_model, examples, candidates)
    first_probs = {str(k): v for k, v in probs[0].items()}
    last_probs = {str(k): v for k, v in probs[1].items()}
    pair_probs = {}
    offset = 2
    for first_index, second_index in PAIR_INDICES:
        a, b = first_index + 1, second_index + 1
        forward = probs[offset]
        offset += 1
        p_a_before_b = float(forward[1])
        if PAIRWISE_BIDIRECTIONAL:
            reverse = probs[offset]
            offset += 1
            p_a_before_b = 0.5 * (p_a_before_b + float(reverse[2]))
        pair_probs[f"{a}>{b}"] = p_a_before_b
        pair_probs[f"{b}>{a}"] = 1.0 - p_a_before_b

    candidates24 = []
    for order in PERMUTATIONS:
        score = structured_score(first_probs, last_probs, pair_probs, order)
        candidates24.append({"order": list(order), "structured_score": score})
    candidates24 = sorted(candidates24, key=lambda item: item["structured_score"], reverse=True)
    for rank, item in enumerate(candidates24, start=1):
        item["rank"] = rank

    gold_order = list(row["gold_order"]) if has_gold else None
    gold_rank = None
    for item in candidates24:
        item["is_gold"] = None if gold_order is None else (item["order"] == gold_order)
        if item["is_gold"]:
            gold_rank = item["rank"]
    return {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "gold_order": gold_order,
        "gold_rank": gold_rank,
        "first_probs": first_probs,
        "last_probs": last_probs,
        "pair_probs": pair_probs,
        "candidate_orders": candidates24,
        "baseline_order": candidates24[0]["order"],
        "top1_top2_gap": float(candidates24[0]["structured_score"] - candidates24[1]["structured_score"]),
    }


def baseline_cache_path(split_name, rows):
    manifest = {
        "kind": "baseline_structured",
        "split": split_name,
        "ids_hash": ids_hash(rows["Id"].astype(str).tolist()),
        "adapter": adapter_fingerprint(INITIAL_ADAPTER_DIR),
        "model_repo_id": MODEL_REPO_ID,
        "prompt_version": PROMPT_VERSION,
        "min_pixels": MIN_PIXELS,
        "max_pixels": MAX_PIXELS,
        "pairwise_bidirectional": PAIRWISE_BIDIRECTIONAL,
        "alpha": STRUCTURED_ALPHA,
        "beta": STRUCTURED_BETA,
        "gamma": STRUCTURED_GAMMA,
    }
    key = sha16(json.dumps(manifest, sort_keys=True, ensure_ascii=False))
    return os.path.join(BASELINE_CACHE_DIR, key, f"{split_name}_baseline.jsonl"), os.path.join(BASELINE_CACHE_DIR, key, "manifest.json"), manifest


def generate_baseline_cache(active_model, split_name, rows, image_root):
    path, manifest_path, manifest = baseline_cache_path(split_name, rows)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    expected_ids = rows["Id"].astype(str).tolist()
    if os.path.exists(path) and os.path.exists(manifest_path) and load_json(manifest_path) == manifest:
        cached = read_jsonl(path)
        if [r["sample_id"] for r in cached] == expected_ids:
            print("[CACHE]", path)
            return cached
    partial = path + ".partial"
    done = []
    if os.path.exists(partial) and os.path.exists(manifest_path) and load_json(manifest_path) == manifest:
        done = read_jsonl(partial)
        if [r["sample_id"] for r in done] != expected_ids[:len(done)]:
            done = []
    if not done:
        if os.path.exists(partial):
            os.remove(partial)
        save_json(manifest, manifest_path)
    done_ids = {r["sample_id"] for r in done}
    records = list(done)
    remaining = rows[~rows["Id"].astype(str).isin(done_ids)]
    for _, row in tqdm(remaining.iterrows(), total=len(remaining), desc=f"baseline {split_name}"):
        record = build_baseline_record(active_model, row, image_root, has_gold="Answer" in row)
        append_jsonl(record, partial)
        records.append(record)
    os.replace(partial, path)
    save_json(manifest, manifest_path)
    return records


In [ ]:
# 6) Hypothesis/evidence/verifier prompts and caches
def fallback_hypothesis(sentence):
    # Parse failures must not create fake evidence. They disable hypothesis-side scoring.
    return {"hypotheses": [], "uncertain_fields": [], "fallback_used": True}


def normalize_hypothesis(obj, sentence):
    if not isinstance(obj, dict) or not isinstance(obj.get("hypotheses"), list):
        obj = fallback_hypothesis(sentence)
    normalized = {"hypotheses": [], "uncertain_fields": obj.get("uncertain_fields", []) if isinstance(obj.get("uncertain_fields"), list) else []}
    for h_index, hyp in enumerate(obj.get("hypotheses", [])[:K_TEXT_HYPOTHESES], start=1):
        if not isinstance(hyp, dict):
            continue
        events = []
        for e_index, event in enumerate(hyp.get("events", [])[:4], start=1):
            if not isinstance(event, dict):
                continue
            events.append({
                "event_id": str(event.get("event_id") or f"E{e_index}"),
                "subject": str(event.get("subject", "")),
                "action": str(event.get("action", "")),
                "objects": [str(x) for x in event.get("objects", [])] if isinstance(event.get("objects", []), list) else [],
                "states": [str(x) for x in event.get("states", [])] if isinstance(event.get("states", []), list) else [],
                "camera": str(event.get("camera", "")),
            })
        if len(events) < 2:
            continue
        constraints = hyp.get("temporal_constraints", [])
        constraints = [c for c in constraints if isinstance(c, list) and len(c) >= 3]
        normalized["hypotheses"].append({"hypothesis_id": str(hyp.get("hypothesis_id") or f"H{h_index}"), "events": events, "temporal_constraints": constraints})
    if not normalized["hypotheses"]:
        return fallback_hypothesis(sentence)
    return normalized


def text_hypothesis_prompt(sentence):
    return f'''Analyze only the caption below. Do not infer image numbers or use any prior order.

Caption:
{sentence}

List the events strictly in chronological order, from the earliest event to the latest event.

Return compact JSON with this schema:
{{
  "hypotheses": [
    {{
      "hypothesis_id": "H1",
      "events": [
        {{"event_id": "E1", "subject": "", "action": "", "objects": [], "states": [], "camera": ""}}
      ],
      "temporal_constraints": [["E1", "before", "E2"]]
    }}
  ],
  "uncertain_fields": []
}}

Extract between 2 and 4 temporally meaningful events. Merge minor simultaneous actions rather than producing more than 4 events. Use at most {K_TEXT_HYPOTHESES} hypotheses. Use a second hypothesis only for real ambiguity. No chain-of-thought. JSON only.'''


def generate_text_hypothesis(active_model, row):
    sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
    raw = generate_text(active_model, user_message_text_only(text_hypothesis_prompt(sentence)), image_paths=[], max_new_tokens=MAX_NEW_TOKENS_TEXT)
    parsed = safe_json_loads(raw, fallback_hypothesis(sentence))
    return {"sample_id": str(row["Id"]), "sentence": sentence, "raw_text": raw, "hypothesis": normalize_hypothesis(parsed, sentence)}


def fallback_visual():
    return {
        "images": {str(i): {"subjects": [], "objects": [], "actions": [], "states": [], "camera": [], "observations": []} for i in range(1, 5)},
        "transitions": {},
    }


def normalize_visual(obj):
    if not isinstance(obj, dict):
        obj = fallback_visual()
    out = fallback_visual()
    images = obj.get("images", {})
    if isinstance(images, dict):
        for i in range(1, 5):
            source = images.get(str(i), {})
            if isinstance(source, dict):
                out["images"][str(i)] = {
                    key: [str(x) for x in source.get(key, [])] if isinstance(source.get(key, []), list) else []
                    for key in ["subjects", "objects", "actions", "states", "camera", "observations"]
                }
    # Transition extraction is intentionally disabled for the first pilot.
    out["transitions"] = {}
    return out


def visual_evidence_prompt(sentence=None):
    return f'''Observe the four images independently. Do not use the caption, do not use any baseline order, do not infer chronology, and do not decide the final order.

Extract observable facts only:
- subjects
- objects
- actions/postures
- states
- camera/framing
- concrete observations visible in a single image

Return JSON:
{{
  "images": {{
    "1": {{"subjects": [], "objects": [], "actions": [], "states": [], "camera": [], "observations": []}},
    "2": {{"subjects": [], "objects": [], "actions": [], "states": [], "camera": [], "observations": []}},
    "3": {{"subjects": [], "objects": [], "actions": [], "states": [], "camera": [], "observations": []}},
    "4": {{"subjects": [], "objects": [], "actions": [], "states": [], "camera": [], "observations": []}}
  }}
}}
JSON only.'''


def generate_visual_evidence(active_model, row, image_root):
    image_paths = row_image_paths(row, image_root)
    raw = generate_text(active_model, user_message_with_images(4, visual_evidence_prompt()), image_paths=image_paths, max_new_tokens=MAX_NEW_TOKENS_VISUAL)
    parsed = safe_json_loads(raw, fallback_visual())
    return {"sample_id": str(row["Id"]), "raw_text": raw, "visual": normalize_visual(parsed)}


def cache_records(active_model, split_name, rows, image_root, cache_dir, kind, maker):
    manifest = {
        "kind": kind,
        "split": split_name,
        "ids_hash": ids_hash(rows["Id"].astype(str).tolist()),
        "adapter": adapter_fingerprint(INITIAL_ADAPTER_DIR),
        "prompt_version": PROMPT_VERSION,
        "min_pixels": MIN_PIXELS,
        "max_pixels": MAX_PIXELS,
    }
    key = sha16(json.dumps(manifest, sort_keys=True, ensure_ascii=False))
    path = os.path.join(cache_dir, key, f"{split_name}_{kind}.jsonl")
    manifest_path = os.path.join(cache_dir, key, "manifest.json")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    expected_ids = rows["Id"].astype(str).tolist()
    if os.path.exists(path) and os.path.exists(manifest_path) and load_json(manifest_path) == manifest:
        cached = read_jsonl(path)
        if [r["sample_id"] for r in cached] == expected_ids:
            print("[CACHE]", path)
            return cached
    partial = path + ".partial"
    done = []
    if os.path.exists(partial) and os.path.exists(manifest_path) and load_json(manifest_path) == manifest:
        done = read_jsonl(partial)
        if [r["sample_id"] for r in done] != expected_ids[:len(done)]:
            done = []
    if not done:
        if os.path.exists(partial):
            os.remove(partial)
        save_json(manifest, manifest_path)
    done_ids = {r["sample_id"] for r in done}
    records = list(done)
    remaining = rows[~rows["Id"].astype(str).isin(done_ids)]
    for _, row in tqdm(remaining.iterrows(), total=len(remaining), desc=f"{kind} {split_name}"):
        record = maker(active_model, row, image_root)
        append_jsonl(record, partial)
        records.append(record)
    os.replace(partial, path)
    save_json(manifest, manifest_path)
    return records


In [ ]:
# 7) Joint scoring, dispute selection, blind verification, conservative revision
STOPWORDS = {"the", "a", "an", "and", "or", "to", "of", "in", "on", "with", "while", "is", "are", "person", "people", "man", "woman", "water", "image"}
ACTION_SYNONYMS = {
    "glide": {"glide", "glides", "moving", "move", "moves", "ride", "rides"},
    "closer": {"closer", "larger", "nearer", "approaches", "toward", "zoom_in"},
    "wider": {"wider", "wide", "widens", "farther", "distant", "zoom_out"},
    "hold": {"hold", "holding", "holds", "grip", "gripping"},
    "open": {"open", "opened", "opens", "unfold", "unfolded"},
}


def normalize_phrases(text):
    text = str(text).lower()
    replacements = {
        "zoomed out": "zoom_out",
        "zooms out": "zoom_out",
        "zooming out": "zoom_out",
        "zoom out": "zoom_out",
        "zoomed in": "zoom_in",
        "zooms in": "zoom_in",
        "zooming in": "zoom_in",
        "zoom in": "zoom_in",
    }
    for source, target in replacements.items():
        text = text.replace(source, target)
    return text


def tokens(text):
    raw = [t for t in re.findall(r"[a-zA-Z0-9_]+", normalize_phrases(text)) if len(t) >= 3 and t not in STOPWORDS]
    expanded = set(raw)
    for token in raw:
        for _, group in ACTION_SYNONYMS.items():
            if token in group:
                expanded |= group
    return expanded


def field_tokens(values):
    if isinstance(values, list):
        return tokens(" ".join(str(v) for v in values))
    return tokens(values)


def event_field_tokens(event, field):
    if field == "subject":
        return field_tokens(event.get("subject", ""))
    if field == "action":
        return field_tokens(event.get("action", ""))
    if field == "object_state":
        return field_tokens(event.get("objects", [])) | field_tokens(event.get("states", []))
    if field == "camera":
        return field_tokens(event.get("camera", ""))
    return set()


def image_field_tokens(visual, image_number, field):
    data = visual.get("images", {}).get(str(image_number), {})
    if field == "subject":
        return field_tokens(data.get("subjects", []))
    if field == "action":
        return field_tokens(data.get("actions", [])) | field_tokens(data.get("observations", []))
    if field == "object_state":
        return field_tokens(data.get("objects", [])) | field_tokens(data.get("states", []))
    if field == "camera":
        return field_tokens(data.get("camera", []))
    return set()


def overlap_score(a, b):
    if not a or not b:
        return 0.0
    return float(len(a & b) / max(len(a), 1))


def event_image_component_scores(event, visual, image_number):
    subject = overlap_score(event_field_tokens(event, "subject"), image_field_tokens(visual, image_number, "subject"))
    action = overlap_score(event_field_tokens(event, "action"), image_field_tokens(visual, image_number, "action"))
    object_state = overlap_score(event_field_tokens(event, "object_state"), image_field_tokens(visual, image_number, "object_state"))
    camera = overlap_score(event_field_tokens(event, "camera"), image_field_tokens(visual, image_number, "camera"))
    total = 0.15 * subject + 0.40 * action + 0.20 * object_state + 0.25 * camera
    return {"subject": subject, "action": action, "object_state": object_state, "camera": camera, "total": total}


def monotonic_assignments(num_events, num_frames=4):
    if num_events <= 0:
        return []
    if num_events == 1:
        return [(0,) * num_frames]
    # Every event appears at least once when possible; adjacent frames may share an event.
    assignments = []
    def rec(prefix, remaining):
        if len(prefix) == num_frames:
            if set(prefix) >= set(range(min(num_events, num_frames))):
                assignments.append(tuple(prefix))
            return
        start = prefix[-1] if prefix else 0
        for value in range(start, num_events):
            rec(prefix + [value], remaining - 1)
    rec([], num_frames)
    if not assignments:
        assignments = [tuple(min(i, num_events - 1) for i in range(num_frames))]
    return assignments


def transition_keywords_between_events(events, left_event_idx, right_event_idx):
    if left_event_idx == right_event_idx:
        return set()
    left = events[left_event_idx]
    right = events[right_event_idx]
    return (
        event_field_tokens(left, "action")
        | event_field_tokens(right, "action")
        | event_field_tokens(left, "camera")
        | event_field_tokens(right, "camera")
        | event_field_tokens(left, "object_state")
        | event_field_tokens(right, "object_state")
    )


def visual_transition_tokens(visual, left_image, right_image):
    transitions = visual.get("transitions", {})
    values = transitions.get(f"{left_image}->{right_image}", {}).get("changes", [])
    return field_tokens(values)


def hypothesis_order_scores(hypothesis, visual, order, preset):
    events = hypothesis.get("events", [])
    if not events:
        return {
            "grounding": 0.0,
            "transition": 0.0,
            "temporal": 0.0,
            "grounding_components": {},
            "assignment": [],
            "status": "fallback_or_empty",
        }
    events = events[:4]
    event_scores = {
        event_idx: {image: event_image_component_scores(event, visual, image) for image in [1, 2, 3, 4]}
        for event_idx, event in enumerate(events)
    }
    best = None
    for assignment in monotonic_assignments(len(events), num_frames=4):
        grounding_terms = []
        transition_terms = []
        temporal_terms = []
        for frame_idx, image in enumerate(order):
            event_idx = assignment[frame_idx]
            grounding_terms.append(event_scores[event_idx][image]["total"])
        for frame_idx, (left_image, right_image) in enumerate(zip(order[:-1], order[1:])):
            left_event_idx, right_event_idx = assignment[frame_idx], assignment[frame_idx + 1]
            expected = transition_keywords_between_events(events, left_event_idx, right_event_idx)
            observed = visual_transition_tokens(visual, left_image, right_image)
            if expected or observed:
                transition_terms.append(overlap_score(expected, observed))
        event_to_first_image = {}
        for frame_idx, image in enumerate(order):
            event_to_first_image.setdefault(assignment[frame_idx], image)
        image_position = {image: idx for idx, image in enumerate(order)}
        for constraint in hypothesis.get("temporal_constraints", []):
            if len(constraint) < 3 or constraint[1] != "before":
                continue
            event_ids = [event.get("event_id") for event in events]
            if constraint[0] not in event_ids or constraint[2] not in event_ids:
                continue
            left_idx, right_idx = event_ids.index(constraint[0]), event_ids.index(constraint[2])
            if left_idx in event_to_first_image and right_idx in event_to_first_image:
                temporal_terms.append(float(image_position[event_to_first_image[left_idx]] < image_position[event_to_first_image[right_idx]]))
        components = {
            "grounding": float(np.mean(grounding_terms)) if grounding_terms else 0.0,
            "transition": float(np.mean(transition_terms)) if transition_terms else 0.0,
            "temporal": float(np.mean(temporal_terms)) if temporal_terms else 0.0,
            "assignment": list(assignment),
        }
        selection_score = (
            preset["grounding"] * components["grounding"]
            + preset["transition"] * components["transition"]
            + preset["temporal"] * components["temporal"]
        )
        if best is None or selection_score > best["raw"]:
            best = components | {"raw": selection_score}
    return {
        "grounding": best["grounding"],
        "transition": best["transition"],
        "temporal": best["temporal"],
        "assignment": best["assignment"],
        "grounding_components": {},
        "status": "ok",
    }


def minmax(values, min_span=0.03):
    values = np.asarray(values, dtype=np.float64)
    span = float(values.max() - values.min()) if len(values) else 0.0
    if span < min_span:
        return np.zeros_like(values)
    return (values - float(values.min())) / span


def joint_score_record(baseline_record, text_record, visual_record, preset):
    visual = visual_record["visual"]
    hypotheses = text_record["hypothesis"].get("hypotheses", [])
    items = []
    for item in baseline_record["candidate_orders"]:
        order = item["order"]
        best_components = {"grounding": 0.0, "transition": 0.0, "temporal": 0.0, "hypothesis_id": "", "assignment": [], "status": "no_hypothesis"}
        best_raw = -1e9
        for hypothesis in hypotheses:
            components = hypothesis_order_scores(hypothesis, visual, order, preset)
            selection_score = (
                preset["grounding"] * components["grounding"]
                + preset["transition"] * components["transition"]
                + preset["temporal"] * components["temporal"]
            )
            if selection_score > best_raw:
                best_raw = selection_score
                best_components = components | {"hypothesis_id": hypothesis.get("hypothesis_id", ""), "hypothesis": hypothesis}
        items.append({
            "order": order,
            "structured_score": float(item["structured_score"]),
            "grounding_raw": float(best_components["grounding"]),
            "transition_raw": float(best_components["transition"]),
            "temporal_raw": float(best_components["temporal"]),
            "components": best_components,
        })
    structured_values = [item["structured_score"] for item in items]
    grounding_values = [item["grounding_raw"] for item in items]
    transition_values = [item["transition_raw"] for item in items]
    temporal_values = [item["temporal_raw"] for item in items]
    structured_span = float(max(structured_values) - min(structured_values)) if structured_values else 0.0
    grounding_span = float(max(grounding_values) - min(grounding_values)) if grounding_values else 0.0
    structured_norm = minmax(structured_values, min_span=0.03)
    grounding_norm = minmax(grounding_values, min_span=0.03)
    transition_norm = minmax(transition_values, min_span=0.03)
    temporal_norm = minmax(temporal_values, min_span=0.03)
    for idx, item in enumerate(items):
        item["structured_norm"] = float(structured_norm[idx])
        item["grounding_norm"] = float(grounding_norm[idx])
        item["transition_norm"] = float(transition_norm[idx])
        item["temporal_norm"] = float(temporal_norm[idx])
        item["grounding_raw_span"] = grounding_span
        item["structured_raw_span"] = structured_span
        item["joint_score"] = float(
            preset["structured"] * item["structured_norm"]
            + preset["grounding"] * item["grounding_norm"]
            + preset["transition"] * item["transition_norm"]
            + preset["temporal"] * item["temporal_norm"]
        )
    return sorted(items, key=lambda item: item["joint_score"], reverse=True)


def differing_pairs(order_a, order_b):
    rank_a = {image: idx for idx, image in enumerate(order_a)}
    rank_b = {image: idx for idx, image in enumerate(order_b)}
    out = []
    for x, y in itertools.combinations([1, 2, 3, 4], 2):
        if (rank_a[x] < rank_a[y]) != (rank_b[x] < rank_b[y]):
            out.append((x, y))
    return out


def event_summary(event):
    if not isinstance(event, dict):
        return {"action": "", "state": "", "camera": "", "objects": []}
    return {
        "action": str(event.get("action", "")),
        "state": "; ".join(str(x) for x in event.get("states", [])) if isinstance(event.get("states", []), list) else str(event.get("states", "")),
        "camera": str(event.get("camera", "")),
        "objects": [str(x) for x in event.get("objects", [])] if isinstance(event.get("objects", []), list) else [],
    }


def pair_event_focus(joint_item, order, earlier, later):
    comp = joint_item.get("components", {})
    hypothesis = comp.get("hypothesis", {}) if isinstance(comp.get("hypothesis", {}), dict) else {}
    events = hypothesis.get("events", []) if isinstance(hypothesis.get("events", []), list) else []
    assignment = comp.get("assignment", [])
    rank = {image: idx for idx, image in enumerate(order)}
    if not events or len(assignment) <= max(rank[earlier], rank[later]):
        return {"earlier_event": {}, "later_event": {}}
    earlier_idx = int(assignment[rank[earlier]])
    later_idx = int(assignment[rank[later]])
    return {
        "earlier_event": event_summary(events[earlier_idx] if earlier_idx < len(events) else {}),
        "later_event": event_summary(events[later_idx] if later_idx < len(events) else {}),
    }


def select_disputes(baseline_record, joint_ranked, max_verifications):
    baseline_order = baseline_record["baseline_order"]
    top = joint_ranked[0]["order"]
    if top == baseline_order:
        return []
    pairs = differing_pairs(baseline_order, top)
    pair_probs = baseline_record["pair_probs"]

    def priority(pair):
        a, b = pair
        return abs(float(pair_probs.get(f"{a}>{b}", 0.5)) - 0.5)

    disputes = []
    rank_top = {image: idx for idx, image in enumerate(top)}
    for a, b in sorted(pairs, key=priority)[:max_verifications]:
        earlier, later = (a, b) if rank_top[a] < rank_top[b] else (b, a)
        focus = pair_event_focus(joint_ranked[0], top, earlier, later)
        disputes.append({
            "image_a": int(earlier),
            "image_b": int(later),
            "canonical_pair": sorted([int(a), int(b)]),
            "compare_source": "baseline_vs_joint_top",
            "evidence_type": "event_focused_blind_pair",
            "earlier_event": focus.get("earlier_event", {}),
            "later_event": focus.get("later_event", {}),
            "focus_hash": sha16(focus),
            "hypothesis_id": joint_ranked[0].get("components", {}).get("hypothesis_id", ""),
            "assignment": joint_ranked[0].get("components", {}).get("assignment", []),
        })
    return disputes


def format_event_focus(dispute):
    if not dispute:
        return ""
    earlier = dispute.get("earlier_event", {})
    later = dispute.get("later_event", {})
    if not earlier and not later:
        return ""
    return f"""
Caption-derived event descriptions, without assigning them to Image 1 or Image 2:
Earlier event: action={earlier.get('action', '')}; state={earlier.get('state', '')}; camera={earlier.get('camera', '')}; objects={earlier.get('objects', [])}
Later event: action={later.get('action', '')}; state={later.get('state', '')}; camera={later.get('camera', '')}; objects={later.get('objects', [])}
Choose which image better matches the earlier event. Do not assume either image is earlier from its label.
"""


def verifier_prompt(sentence, a, b, dispute=None):
    hint = format_event_focus(dispute)
    return f'''Caption:
{sentence}
{hint}
You will see two images labeled Image 1 and Image 2. Decide which image occurs earlier in time.
Answer only 1 if Image 1 is earlier, or 2 if Image 2 is earlier.'''


@torch.inference_mode()
def verify_pair(active_model, row, image_root, a, b, dispute=None):
    image_paths_all = row_image_paths(row, image_root)
    sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
    prompts = []
    image_batches = []
    for x, y in [(a, b), (b, a)]:
        messages = user_message_with_images(2, verifier_prompt(sentence, x, y, dispute=dispute))
        prompts.append(chat_prompt(messages, add_generation_prompt=True))
        image_batches.append([load_rgb(image_paths_all[x - 1]), load_rgb(image_paths_all[y - 1])])
    old_padding_side = processor.tokenizer.padding_side
    try:
        processor.tokenizer.padding_side = "right"
        inputs = processor(text=prompts, images=image_batches, padding=True, return_tensors="pt")
        inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = active_model(**inputs)
        probs = []
        for row_idx in range(2):
            last_pos = int(inputs["attention_mask"][row_idx].sum().item()) - 1
            logits = outputs.logits[row_idx, last_pos]
            p = torch.softmax(logits[[DIGIT_TOKEN_IDS[1], DIGIT_TOKEN_IDS[2]]].float(), dim=-1).detach().cpu().numpy()
            probs.append(float(p[0]))
    finally:
        processor.tokenizer.padding_side = old_padding_side
    forward = probs[0]
    reverse_support = 1.0 - probs[1]
    combined = 0.5 * (forward + reverse_support)
    agreement = 1.0 - abs(forward - reverse_support)
    return {
        "claim_direction": [int(a), int(b)],
        "canonical_pair": sorted([int(a), int(b)]),
        "forward_probability": forward,
        "reverse_probability": reverse_support,
        "combined_probability": combined,
        "forward_reverse_agreement": agreement,
        "position_consistent": bool(combined >= 0.5),
    }


def conservative_decision(baseline_record, joint_ranked, verifications, switch_margin, order_switch_gain):
    baseline_order = baseline_record["baseline_order"]
    joint_top = joint_ranked[0]["order"]
    if joint_top == baseline_order:
        return baseline_order, "KEEP_BASELINE_JOINT_SAME", False
    changed_pairs = differing_pairs(baseline_order, joint_top)
    if len(changed_pairs) > KENDALL_SWITCH_MAX:
        return baseline_order, "KEEP_TOO_MANY_RELATIONS_CHANGED", False
    if len(verifications) < len(changed_pairs):
        return baseline_order, "KEEP_NOT_ALL_RELATIONS_VERIFIED", False
    all_strong = all(
        v["position_consistent"]
        and float(v["forward_probability"]) >= 0.5 + switch_margin
        and float(v["reverse_probability"]) >= 0.5 + switch_margin
        and float(v["forward_reverse_agreement"]) >= FORWARD_REVERSE_AGREEMENT_MIN
        for v in verifications
    )
    joint_gain = float(joint_ranked[0]["joint_score"] - next(item["joint_score"] for item in joint_ranked if item["order"] == baseline_order))
    should_switch = all_strong and joint_gain >= order_switch_gain
    if should_switch:
        return joint_top, "REVISE", True
    return baseline_order, "KEEP_CONSERVATIVE", False


In [ ]:
# 8) End-to-end run functions and evaluation
def records_by_id(records):
    return {record["sample_id"]: record for record in records}


def kendall_distance(order_a, order_b):
    return len(differing_pairs(order_a, order_b))


def evaluate_predictions(rows):
    df = pd.DataFrame(rows)
    baseline_correct = df["baseline_exact_match"].eq(1.0)
    joint_correct = df["joint_exact_match"].eq(1.0)
    c_correct = df["c_exact_match"].eq(1.0)
    corrected = (~baseline_correct & c_correct)
    introduced = (baseline_correct & ~c_correct)
    preserved = (baseline_correct & c_correct)
    joint_corrected = (~baseline_correct & joint_correct)
    joint_introduced = (baseline_correct & ~joint_correct)
    verifier_blocked_bad_revision = (joint_introduced & c_correct)
    verifier_blocked_good_revision = (joint_corrected & ~c_correct)
    summary = {
        "count": int(len(df)),
        "baseline_exact_match": float(df["baseline_exact_match"].mean()),
        "joint_exact_match": float(df["joint_exact_match"].mean()),
        "c_exact_match": float(df["c_exact_match"].mean()),
        "baseline_pair_accuracy": float(df["baseline_pair_accuracy"].mean()),
        "joint_pair_accuracy": float(df["joint_pair_accuracy"].mean()),
        "c_pair_accuracy": float(df["c_pair_accuracy"].mean()),
        "baseline_position_accuracy": float(df["baseline_position_accuracy"].mean()),
        "joint_position_accuracy": float(df["joint_position_accuracy"].mean()),
        "c_position_accuracy": float(df["c_position_accuracy"].mean()),
        "joint_corrected_errors": int(joint_corrected.sum()),
        "joint_introduced_errors": int(joint_introduced.sum()),
        "verifier_blocked_bad_revision": int(verifier_blocked_bad_revision.sum()),
        "verifier_blocked_good_revision": int(verifier_blocked_good_revision.sum()),
        "corrected_errors": int(corrected.sum()),
        "introduced_errors": int(introduced.sum()),
        "net_corrections": int(corrected.sum() - introduced.sum()),
        "recovery_rate": float(corrected.sum() / max((~baseline_correct).sum(), 1)),
        "preservation_rate": float(preserved.sum() / max(baseline_correct.sum(), 1)),
        "verification_rate": float(df["verified"].mean()),
        "revision_rate": float(df["revised"].mean()),
        "avg_extra_forwards": float(df["extra_forwards"].mean()),
        "avg_kendall_baseline_joint": float(df["kendall_baseline_joint"].mean()),
        "joint_baseline_disagreement_rate": float(df["kendall_baseline_joint"].gt(0).mean()),
        "grounding_raw_span": float(df["grounding_raw_span"].mean()) if "grounding_raw_span" in df else 0.0,
        "structured_raw_span": float(df["structured_raw_span"].mean()) if "structured_raw_span" in df else 0.0,
    }
    return summary, df


def image_file_fingerprint(path):
    try:
        stat = os.stat(path)
        return {"name": os.path.basename(path), "size": int(stat.st_size), "mtime": int(stat.st_mtime)}
    except FileNotFoundError:
        return {"name": os.path.basename(path), "missing": True}


def sample_input_fingerprint(row, image_root):
    image_paths = row_image_paths(row, image_root)
    sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
    return sha16({
        "caption": sentence,
        "images": [image_file_fingerprint(path) for path in image_paths],
    })


def verification_cache_key(sample_id, dispute):
    canonical = "-".join(str(x) for x in sorted(dispute["canonical_pair"]))
    return sha16({
        "prompt_version": PROMPT_VERSION,
        "adapter": adapter_fingerprint(INITIAL_ADAPTER_DIR),
        "model_id": str(MODEL_ID),
        "min_pixels": MIN_PIXELS,
        "max_pixels": MAX_PIXELS,
        "sample_id": str(sample_id),
        "sample_input_fingerprint": dispute.get("sample_input_fingerprint", ""),
        "canonical_pair": canonical,
        "focus_hash": dispute.get("focus_hash", ""),
        "verifier_prompt_hash": sha16(verifier_prompt("", 1, 2, dispute)),
    })


def orient_verification(record, dispute):
    desired = [int(dispute["image_a"]), int(dispute["image_b"])]
    stored = [int(x) for x in record.get("claim_direction", desired)]
    out = dict(record)
    out["requested_claim_direction"] = desired
    if stored == desired:
        return out
    if stored == [desired[1], desired[0]]:
        out["forward_probability"] = 1.0 - float(record["reverse_probability"])
        out["reverse_probability"] = 1.0 - float(record["forward_probability"])
        out["combined_probability"] = 1.0 - float(record["combined_probability"])
        out["position_consistent"] = bool(out["combined_probability"] >= 0.5)
        return out
    raise ValueError({"stored_direction": stored, "desired_direction": desired})


def load_verification_cache(path):
    cache = {}
    if os.path.exists(path):
        for record in read_jsonl(path):
            cache[record["cache_key"]] = record
    return cache


def append_verification_cache(path, record):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def verification_cache_path(split_name, rows, presets):
    preset_names = "_".join(p["name"] for p in presets)
    manifest = {
        "split": split_name,
        "prompt": PROMPT_VERSION,
        "adapter": adapter_fingerprint(INITIAL_ADAPTER_DIR),
        "model_id": str(MODEL_ID),
        "min_pixels": MIN_PIXELS,
        "max_pixels": MAX_PIXELS,
        "preset_names": preset_names,
        "ids": rows["Id"].astype(str).tolist(),
    }
    return os.path.join(VERIFY_CACHE_DIR, f"{split_name}_{sha16(manifest)}_verification.jsonl")


def prepare_pipeline_state(active_model, split_name, rows, baseline_records, text_records, visual_records, presets, max_verifications):
    base_by_id = records_by_id(baseline_records)
    text_by_id = records_by_id(text_records)
    visual_by_id = records_by_id(visual_records)
    row_by_id = {str(row["Id"]): row for _, row in rows.iterrows()}
    state = {}
    for preset in presets:
        preset_state = {}
        for sample_id, baseline in tqdm(base_by_id.items(), desc=f"joint once {split_name} {preset['name']}"):
            joint_ranked = joint_score_record(baseline, text_by_id[sample_id], visual_by_id[sample_id], preset)
            disputes = select_disputes(baseline, joint_ranked, max_verifications=max_verifications)
            sample_fp = sample_input_fingerprint(row_by_id[sample_id], TRAIN_IMAGE_DIR)
            for dispute in disputes:
                dispute["sample_input_fingerprint"] = sample_fp
            preset_state[sample_id] = {"joint_ranked": joint_ranked, "disputes": disputes}
        state[preset["name"]] = preset_state
    cache_path = verification_cache_path(split_name, rows, presets)
    verification_cache = load_verification_cache(cache_path)
    print("verification cache loaded:", cache_path, "records:", len(verification_cache))
    return state, verification_cache


def sample_needs_verification(baseline, sample_state, gap_threshold, verify_margin):
    joint_ranked = sample_state["joint_ranked"]
    joint_top = joint_ranked[0]["order"]
    baseline_order = baseline["baseline_order"]
    if joint_top == baseline_order:
        return False
    baseline_gap = abs(float(baseline["top1_top2_gap"]))
    joint_gap = abs(float(joint_ranked[0]["joint_score"] - joint_ranked[1]["joint_score"]))
    return baseline_gap <= gap_threshold or joint_gap <= verify_margin


def ensure_verifications(active_model, split_name, rows, baseline_records, state, verification_cache, presets, gap_threshold, verify_margin, max_verifications):
    base_by_id = records_by_id(baseline_records)
    row_by_id = {str(row["Id"]): row for _, row in rows.iterrows()}
    cache_path = verification_cache_path(split_name, rows, presets)
    targets = []
    seen = set()
    for preset in presets:
        preset_state = state[preset["name"]]
        for sample_id, baseline in base_by_id.items():
            sample_state = preset_state[sample_id]
            if not sample_needs_verification(baseline, sample_state, gap_threshold, verify_margin):
                continue
            for dispute in sample_state["disputes"][:max_verifications]:
                key = verification_cache_key(sample_id, dispute)
                if key not in seen:
                    targets.append((key, sample_id, preset["name"], dispute))
                    seen.add(key)
    started = time.time()
    hits = sum(1 for key, _, _, _ in targets if key in verification_cache)
    new_pairs = 0
    for key, sample_id, preset_name, dispute in tqdm(targets, desc=f"verify {split_name}"):
        if key in verification_cache:
            continue
        canonical_a, canonical_b = sorted(dispute["canonical_pair"])
        record = verify_pair(active_model, row_by_id[sample_id], TRAIN_IMAGE_DIR, canonical_a, canonical_b, dispute=dispute)
        record.update({"cache_key": key, "sample_id": sample_id, "preset": preset_name, "dispute": dispute})
        verification_cache[key] = record
        append_verification_cache(cache_path, record)
        new_pairs += 1
    stats = {
        "policy_required_pairs": int(len(targets)),
        "policy_required_forwards": int(2 * len(targets)),
        "executed_new_pairs": int(new_pairs),
        "executed_new_forwards": int(2 * new_pairs),
        "verification_cache_hits": int(hits),
        "verification_wall_time": float(time.time() - started),
    }
    print("verification stats:", stats)
    return verification_cache, stats


def evaluate_config_from_state(split_name, rows, baseline_records, state, verification_cache, preset_name, gap_threshold, max_verifications, verify_margin, switch_margin, order_switch_gain):
    base_by_id = records_by_id(baseline_records)
    prediction_rows = []
    verify_rows = []
    joint_rows = []
    for sample_id, baseline in base_by_id.items():
        gold = baseline.get("gold_order")
        sample_state = state[preset_name][sample_id]
        joint_ranked = sample_state["joint_ranked"]
        baseline_order = baseline["baseline_order"]
        joint_top = joint_ranked[0]["order"]
        joint_rows.append({"sample_id": sample_id, "preset": preset_name, "ranked": joint_ranked})
        needs_verify = sample_needs_verification(baseline, sample_state, gap_threshold, verify_margin)
        verifications = []
        if needs_verify and max_verifications > 0:
            for dispute in sample_state["disputes"][:max_verifications]:
                key = verification_cache_key(sample_id, dispute)
                if key in verification_cache:
                    oriented = orient_verification(verification_cache[key], dispute)
                    verifications.append(oriented)
                    verify_rows.append(oriented)
        final_order, action, revised = conservative_decision(baseline, joint_ranked, verifications, switch_margin=switch_margin, order_switch_gain=order_switch_gain)
        base_metrics = order_metric_row(baseline_order, gold)
        joint_metrics = order_metric_row(joint_top, gold)
        c_metrics = order_metric_row(final_order, gold)
        top_item = joint_ranked[0]
        prediction_rows.append({
            "sample_id": sample_id,
            "gold_order": gold,
            "baseline_order": baseline_order,
            "joint_top_order": joint_top,
            "c_order": final_order,
            "action": action,
            "verified": float(len(verifications) > 0),
            "verified_count": len(verifications),
            "revised": float(revised),
            "extra_forwards": 2 * len(verifications),
            "kendall_baseline_joint": kendall_distance(baseline_order, joint_top),
            "grounding_raw_span": float(top_item.get("grounding_raw_span", 0.0)),
            "structured_raw_span": float(top_item.get("structured_raw_span", 0.0)),
            **{f"baseline_{k}": v for k, v in base_metrics.items()},
            **{f"joint_{k}": v for k, v in joint_metrics.items()},
            **{f"c_{k}": v for k, v in c_metrics.items()},
        })
    summary, pred_df = evaluate_predictions(prediction_rows)
    summary.update({
        "split": split_name,
        "preset": preset_name,
        "gap_threshold": float(gap_threshold),
        "max_verifications": max_verifications,
        "verify_margin": verify_margin,
        "switch_margin": switch_margin,
        "order_switch_gain": order_switch_gain,
    })
    return summary, pred_df, joint_rows, verify_rows


def run_grid(active_model, split_name, rows, baseline_records, text_records, visual_records, grid_small=False, fixed_gap_threshold=None):
    max_verification_budget = max(MAX_VERIFICATIONS_GRID)
    state, verification_cache = prepare_pipeline_state(active_model, split_name, rows, baseline_records, text_records, visual_records, WEIGHT_PRESETS, max_verification_budget)
    gaps = np.array([abs(float(record["top1_top2_gap"])) for record in baseline_records], dtype=np.float64)
    quantiles = UNCERTAIN_QUANTILE_GRID if not grid_small else [0.30]
    gap_thresholds = [float(np.quantile(gaps, q)) for q in quantiles] if fixed_gap_threshold is None else [float(fixed_gap_threshold)]
    max_verifs = MAX_VERIFICATIONS_GRID if not grid_small else [1]
    verify_margins = VERIFY_MARGIN_GRID if not grid_small else [0.10]
    switch_margins = SWITCH_MARGIN_GRID if not grid_small else [0.15]
    gains = ORDER_SWITCH_GAIN_GRID if not grid_small else [0.05]
    # Verify only the union of samples/pairs that any grid setting can actually use.
    verification_cache, verify_stats = ensure_verifications(
        active_model,
        split_name,
        rows,
        baseline_records,
        state,
        verification_cache,
        WEIGHT_PRESETS,
        gap_threshold=max(gap_thresholds),
        verify_margin=max(verify_margins),
        max_verifications=max(max_verifs),
    )
    summaries = []
    for preset in WEIGHT_PRESETS:
        for gap_threshold in gap_thresholds:
            for mv in max_verifs:
                for vm in verify_margins:
                    for sm in switch_margins:
                        for gain in gains:
                            summary, _, _, _ = evaluate_config_from_state(split_name, rows, baseline_records, state, verification_cache, preset["name"], gap_threshold, mv, vm, sm, gain)
                            summary["uncertain_quantile"] = np.nan if fixed_gap_threshold is not None else quantiles[gap_thresholds.index(gap_threshold)]
                            summary.update(verify_stats)
                            summaries.append(summary)
    summary_df = pd.DataFrame(summaries).sort_values(["c_exact_match", "net_corrections", "preservation_rate", "avg_extra_forwards"], ascending=[False, False, False, True]).reset_index(drop=True)
    return summary_df, state, verification_cache


def evidence_diagnostics(split_name, text_records, visual_records, baseline_records):
    text_parse_ok = [float(not record.get("hypothesis", {}).get("fallback_used", False) and len(record.get("hypothesis", {}).get("hypotheses", [])) > 0) for record in text_records]
    event_counts = [
        sum(len(h.get("events", [])) for h in record.get("hypothesis", {}).get("hypotheses", []))
        for record in text_records
    ]
    visual_empty = []
    transition_counts = []
    for record in visual_records:
        visual = record.get("visual", {})
        image_values = []
        for image_info in visual.get("images", {}).values():
            for key in ["subjects", "objects", "actions", "states", "camera", "observations"]:
                image_values.extend(image_info.get(key, []) if isinstance(image_info.get(key, []), list) else [])
        visual_empty.append(float(len(image_values) == 0))
        transition_counts.append(len(visual.get("transitions", {})))
    kendall_to_baseline = []
    for record in baseline_records:
        # Filled later by C diagnostics; here we at least expose baseline ambiguity.
        kendall_to_baseline.append(float(record.get("top1_top2_gap", 0.0)))
    diagnostics = {
        "split": split_name,
        "text_json_success_rate": float(np.mean(text_parse_ok)) if text_parse_ok else np.nan,
        "text_event_count_mean": float(np.mean(event_counts)) if event_counts else np.nan,
        "visual_empty_rate": float(np.mean(visual_empty)) if visual_empty else np.nan,
        "visual_transition_key_count_mean": float(np.mean(transition_counts)) if transition_counts else np.nan,
        "baseline_top1_top2_gap_mean": float(np.mean(kendall_to_baseline)) if kendall_to_baseline else np.nan,
    }
    return diagnostics


In [ ]:
# 9) Phase 1-3: quick50, tuning150, holdout150
holdout_summary = None
holdout_pred_df = pd.DataFrame()
holdout_joint_rows = []
holdout_verify_rows = []
best = None


def build_split_inputs(active_model, split_name, split_df):
    baseline = generate_baseline_cache(active_model, split_name, split_df, TRAIN_IMAGE_DIR)
    text = cache_records(active_model, split_name, split_df, TRAIN_IMAGE_DIR, TEXT_CACHE_DIR, "text_hypothesis", lambda m, r, root: generate_text_hypothesis(m, r))
    visual = cache_records(active_model, split_name, split_df, TRAIN_IMAGE_DIR, VISUAL_CACHE_DIR, "visual_evidence", generate_visual_evidence)
    diag = evidence_diagnostics(split_name, text, visual, baseline)
    return baseline, text, visual, diag


assert set(quick50_df["Id"]).isdisjoint(set(tuning150_df["Id"]))
assert set(quick50_df["Id"]).isdisjoint(set(holdout150_df["Id"]))
assert set(tuning150_df["Id"]).isdisjoint(set(holdout150_df["Id"]))

eval_model = load_adapter_model(INITIAL_ADAPTER_DIR)
try:
    quick_baseline, quick_text, quick_visual, quick_diag = build_split_inputs(eval_model, "quick50", quick50_df)
    quick_summary, quick_state, quick_verification_cache = run_grid(
        eval_model, "quick50", quick50_df, quick_baseline, quick_text, quick_visual, grid_small=True
    )
    quick_summary.to_csv(os.path.join(EVAL_DIR, "quick50_metrics.csv"), index=False)
    display(pd.DataFrame([quick_diag]))
    display(quick_summary)

    quick_best = quick_summary.iloc[0].to_dict()
    quick_ok = (
        float(quick_diag.get("text_json_success_rate", 0.0)) >= 0.80
        and float(quick_diag.get("visual_empty_rate", 1.0)) <= 0.10
        and float(quick_best.get("grounding_raw_span", 0.0)) >= 0.03
    )
    print("quick_ok:", quick_ok)
    if not quick_ok:
        raise RuntimeError("Quick diagnostics failed. Inspect text/visual outputs before tuning150.")

    tuning_baseline, tuning_text, tuning_visual, tuning_diag = build_split_inputs(eval_model, "tuning150", tuning150_df)
    tuning_summary, tuning_state, tuning_verification_cache = run_grid(
        eval_model, "tuning150", tuning150_df, tuning_baseline, tuning_text, tuning_visual, grid_small=False
    )
    tuning_summary.to_csv(os.path.join(EVAL_DIR, "threshold_grid_tuning.csv"), index=False)
    pd.DataFrame([quick_diag, tuning_diag]).to_csv(os.path.join(EVAL_DIR, "evidence_diagnostics.csv"), index=False)
    display(pd.DataFrame([quick_diag, tuning_diag]))
    display(tuning_summary.head(20))

    best = tuning_summary.iloc[0].to_dict()
    tuning_promising = (
        float(best["c_exact_match"]) > float(best["baseline_exact_match"])
        and int(best["net_corrections"]) > 0
        and float(best["preservation_rate"]) >= 0.95
    )
    print("BEST TUNING")
    print(json.dumps(best, ensure_ascii=False, indent=2))
    print("tuning_promising:", tuning_promising)

    if tuning_promising:
        holdout_baseline, holdout_text, holdout_visual, holdout_diag = build_split_inputs(eval_model, "holdout150", holdout150_df)
        all_diag = pd.DataFrame([quick_diag, tuning_diag, holdout_diag])
        all_diag.to_csv(os.path.join(EVAL_DIR, "evidence_diagnostics.csv"), index=False)
        display(all_diag)

        best_preset = [preset for preset in WEIGHT_PRESETS if preset["name"] == best["preset"]]
        assert len(best_preset) == 1, best["preset"]
        holdout_state, holdout_verification_cache = prepare_pipeline_state(
            eval_model,
            "holdout150",
            holdout150_df,
            holdout_baseline,
            holdout_text,
            holdout_visual,
            best_preset,
            max_verifications=int(best["max_verifications"]),
        )
        holdout_verification_cache, holdout_verify_stats = ensure_verifications(
            eval_model,
            "holdout150",
            holdout150_df,
            holdout_baseline,
            holdout_state,
            holdout_verification_cache,
            best_preset,
            gap_threshold=float(best["gap_threshold"]),
            verify_margin=float(best["verify_margin"]),
            max_verifications=int(best["max_verifications"]),
        )
        holdout_summary, holdout_pred_df, holdout_joint_rows, holdout_verify_rows = evaluate_config_from_state(
            "holdout150",
            holdout150_df,
            holdout_baseline,
            holdout_state,
            holdout_verification_cache,
            best["preset"],
            gap_threshold=float(best["gap_threshold"]),
            max_verifications=int(best["max_verifications"]),
            verify_margin=float(best["verify_margin"]),
            switch_margin=float(best["switch_margin"]),
            order_switch_gain=float(best["order_switch_gain"]),
        )
        holdout_summary.update(holdout_verify_stats)
    else:
        print("Tuning did not clear the gate; holdout is intentionally skipped.")
finally:
    del eval_model
    gc.collect()
    torch.cuda.empty_cache()

if holdout_summary is not None:
    holdout_pred_df.to_csv(os.path.join(EVAL_DIR, "holdout_predictions.csv"), index=False)
    pd.DataFrame([holdout_summary]).to_csv(os.path.join(EVAL_DIR, "holdout_metrics.csv"), index=False)
    write_jsonl(holdout_joint_rows, os.path.join(EVAL_DIR, "joint_candidate_scores.jsonl"))
    write_jsonl(holdout_verify_rows, os.path.join(EVAL_DIR, "verification_diagnostics.jsonl"))

    correction_cases = holdout_pred_df[(holdout_pred_df["baseline_exact_match"] == 0.0) & (holdout_pred_df["c_exact_match"] == 1.0)]
    introduced_cases = holdout_pred_df[(holdout_pred_df["baseline_exact_match"] == 1.0) & (holdout_pred_df["c_exact_match"] == 0.0)]
    correction_cases.to_csv(os.path.join(EVAL_DIR, "correction_cases.csv"), index=False)
    introduced_cases.to_csv(os.path.join(EVAL_DIR, "introduced_error_cases.csv"), index=False)

    print("HOLDOUT")
    print(json.dumps(holdout_summary, ensure_ascii=False, indent=2))
    display(pd.DataFrame([holdout_summary]))


In [ ]:
# 10) Optional Phase 4: training-record construction and pilot training
def make_training_selection_records(train_rows, baseline_records):
    gaps = [float(r["top1_top2_gap"]) for r in baseline_records]
    median_gap = float(np.median(gaps)) if gaps else 0.0
    records = []
    for record in baseline_records:
        gold = record["gold_order"]
        baseline_order = record["baseline_order"]
        exact = order_metric_row(baseline_order, gold)["exact_match"]
        gap = float(record["top1_top2_gap"])
        gold_rank = record.get("gold_rank")
        if exact == 0.0 and gold_rank is not None and gold_rank <= 10:
            category = "recovery"
        elif exact == 1.0 and gap < median_gap:
            category = "hard_preservation"
        elif exact == 1.0:
            category = "easy_preservation"
        else:
            category = "ambiguous"
        records.append({
            "sample_id": record["sample_id"],
            "category": category,
            "gold_order": gold,
            "baseline_order": baseline_order,
            "gold_rank": gold_rank,
            "top1_top2_gap": gap,
            "pair_probs": record["pair_probs"],
        })
    return records


def sample_training_selection(selection_records, seed=SEED):
    rng = np.random.default_rng(seed)
    by_cat = {name: [r for r in selection_records if r["category"] == name] for name in ["recovery", "hard_preservation", "easy_preservation", "ambiguous"]}
    total = len(selection_records)
    targets = {cat: int(total * ratio) for cat, ratio in CATEGORY_RATIOS.items()}
    targets["ambiguous"] = max(0, total - sum(v for k, v in targets.items() if k != "ambiguous"))
    selected = []
    for cat, target in targets.items():
        pool = by_cat[cat]
        rng.shuffle(pool)
        selected.extend(pool[:min(target, len(pool))])
    if not selected:
        selected = selection_records
    rng.shuffle(selected)
    return selected


def gold_pair_target(gold_order, a, b):
    rank = {int(image): idx for idx, image in enumerate(gold_order)}
    return "1" if rank[int(a)] < rank[int(b)] else "2"


def most_informative_pair(selection):
    gold = selection["gold_order"]
    baseline = selection["baseline_order"]
    diff = differing_pairs(gold, baseline)
    if diff:
        pair_probs = selection.get("pair_probs", {})
        return sorted(diff, key=lambda pair: abs(float(pair_probs.get(f"{pair[0]}>{pair[1]}", 0.5)) - 0.5))[0]
    pair_probs = selection.get("pair_probs", {})
    pairs = [(a, b) for a, b in itertools.combinations([1, 2, 3, 4], 2)]
    return sorted(pairs, key=lambda pair: abs(float(pair_probs.get(f"{pair[0]}>{pair[1]}", 0.5)) - 0.5))[0]


def make_training_task_records(selection_records, train_rows, seed=SEED):
    row_by_id = {str(row["Id"]): row for _, row in train_rows.iterrows()}
    rng = random.Random(seed)
    records = []
    for item in selection_records:
        row = row_by_id[item["sample_id"]]
        image_paths = row_image_paths(row, TRAIN_IMAGE_DIR)
        sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
        gold = [int(x) for x in item["gold_order"]]
        baseline = [int(x) for x in item["baseline_order"]]
        a, b = most_informative_pair(item)
        pair_order = [int(a), int(b)]
        if rng.random() < 0.5:
            pair_order = [pair_order[1], pair_order[0]]
        records.append({
            "task_type": "VERIFY_PAIR",
            "sample_id": item["sample_id"],
            "sentence": sentence,
            "image_paths": [image_paths[pair_order[0] - 1], image_paths[pair_order[1] - 1]],
            "pair": pair_order,
            "target": gold_pair_target(gold, pair_order[0], pair_order[1]),
            "category": item["category"],
        })
        records.append({
            "task_type": "FINAL_ORDER",
            "sample_id": item["sample_id"],
            "sentence": sentence,
            "image_paths": image_paths,
            "baseline_order": baseline,
            "target": " ".join(str(x) for x in gold),
            "category": item["category"],
        })
        keep_target = "K" if baseline == gold else "R"
        records.append({
            "task_type": "REVISE_OR_KEEP",
            "sample_id": item["sample_id"],
            "sentence": sentence,
            "image_paths": image_paths,
            "baseline_order": baseline,
            "target": keep_target,
            "category": item["category"],
        })
    return records


def training_instruction(record):
    task = record["task_type"]
    if task == "VERIFY_PAIR":
        a, b = record["pair"]
        return (
            f"<TASK=VERIFY_PAIR>\nCaption:\n{record['sentence']}\n\n"
            "Which image occurs earlier in time? Answer only 1 or 2."
        )
    if task == "FINAL_ORDER":
        return (
            f"<TASK=FINAL_ORDER>\nCaption:\n{record['sentence']}\n\n"
            f"Baseline order: {' '.join(str(x) for x in record['baseline_order'])}\n"
            "Use the images and caption to output the final chronological order. Answer only four numbers separated by spaces."
        )
    if task == "REVISE_OR_KEEP":
        return (
            f"<TASK=REVISE_OR_KEEP>\nCaption:\n{record['sentence']}\n\n"
            f"Baseline order: {' '.join(str(x) for x in record['baseline_order'])}\n"
            "Should the baseline order be kept or revised? Answer only K for keep or R for revise."
        )
    raise ValueError(task)


def training_messages(record, include_answer=False):
    messages = user_message_with_images(len(record["image_paths"]), training_instruction(record))
    if include_answer:
        messages.append({"role": "assistant", "content": str(record["target"])})
    return messages


class MultiHypothesisTrainingDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


class MultiHypothesisTrainingCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids, target):
        ids = input_ids.tolist()
        labels = torch.full_like(input_ids, -100)
        prefix = self.assistant_prefix_ids
        start = None
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            tail = self.tokenizer.decode(ids[-160:], skip_special_tokens=False)
            raise ValueError(f"Assistant prefix not found. target={target!r}, tail={tail!r}")
        target_ids = self.tokenizer.encode(str(target), add_special_tokens=False)
        if ids[start:start + len(target_ids)] != target_ids:
            tail = self.tokenizer.decode(ids[start:start + len(target_ids) + 8], skip_special_tokens=False)
            raise ValueError(f"Unexpected target. target={target!r}, tail={tail!r}")
        labels[start:start + len(target_ids)] = input_ids[start:start + len(target_ids)]
        return labels

    def __call__(self, batch):
        texts, images, targets, task_types = [], [], [], []
        for record in batch:
            texts.append(self.processor.apply_chat_template(training_messages(record, include_answer=True), tokenize=False, add_generation_prompt=False))
            images.append([load_rgb(path) for path in record["image_paths"]])
            targets.append(str(record["target"]))
            task_types.append(record["task_type"])
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        encoded["labels"] = torch.stack([self._mask_prompt(row, target) for row, target in zip(encoded["input_ids"], targets)])
        encoded["task_type"] = task_types
        return encoded


class TaskWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        task_types = inputs.pop("task_type", None)
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
        token_losses = loss_fct(logits.view(-1, logits.size(-1)), shift_labels.view(-1)).view(shift_labels.size())
        mask = shift_labels.ne(-100)
        sample_losses = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        weights = torch.tensor([
            {"VERIFY_PAIR": 1.0, "FINAL_ORDER": 1.0, "REVISE_OR_KEEP": 0.5}.get(task, 1.0)
            for task in task_types
        ], dtype=sample_losses.dtype, device=sample_losses.device)
        loss = (sample_losses * weights).sum() / weights.sum().clamp_min(1e-6)
        return (loss, outputs) if return_outputs else loss


def load_trainable_adapter_model(adapter_dir=INITIAL_ADAPTER_DIR):
    base = load_model_class().from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
        local_files_only=MODEL_LOCAL_FILES_ONLY,
        trust_remote_code=True,
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=True)
    model.config.use_cache = False
    return model


RUN_TRAIN_RECORD_SELECTION = False
RUN_TRAINING_PILOT = False

if RUN_TRAIN_RECORD_SELECTION or RUN_TRAINING_PILOT:
    train_subset = training_df.sample(n=min(TRAIN_ROWS_FOR_RECORDS, len(training_df)), random_state=SEED).reset_index(drop=True).copy()
    model_for_train_cache = load_adapter_model(INITIAL_ADAPTER_DIR)
    try:
        train_baseline = generate_baseline_cache(model_for_train_cache, f"train{len(train_subset)}", train_subset, TRAIN_IMAGE_DIR)
    finally:
        del model_for_train_cache
        gc.collect()
        torch.cuda.empty_cache()
    selection_records = sample_training_selection(make_training_selection_records(train_subset, train_baseline), seed=SEED)
    task_records = make_training_task_records(selection_records, train_subset)
    write_jsonl(selection_records, os.path.join(OUTPUT_DIR, "training_record_selection.jsonl"))
    write_jsonl(task_records, os.path.join(OUTPUT_DIR, "training_task_records.jsonl"))
    dist = pd.Series([r["category"] for r in selection_records]).value_counts(normalize=True).to_dict()
    task_dist = pd.Series([r["task_type"] for r in task_records]).value_counts().to_dict()
    verify_targets = pd.Series([r["target"] for r in task_records if r["task_type"] == "VERIFY_PAIR"]).value_counts(normalize=True).to_dict()
    keep_targets = pd.Series([r["target"] for r in task_records if r["task_type"] == "REVISE_OR_KEEP"]).value_counts(normalize=True).to_dict()
    final_first = pd.Series([str(r["target"]).split()[0] for r in task_records if r["task_type"] == "FINAL_ORDER"]).value_counts(normalize=True).to_dict()
    final_last = pd.Series([str(r["target"]).split()[-1] for r in task_records if r["task_type"] == "FINAL_ORDER"]).value_counts(normalize=True).to_dict()
    diagnostics = {"selection_distribution": dist, "task_distribution": task_dist, "verify_target_distribution": verify_targets, "keep_revise_distribution": keep_targets, "final_first_distribution": final_first, "final_last_distribution": final_last}
    save_json(diagnostics, os.path.join(OUTPUT_DIR, "training_record_distribution.json"))
    print(json.dumps(diagnostics, ensure_ascii=False, indent=2))

    if RUN_TRAINING_PILOT:
        train_model = load_trainable_adapter_model(INITIAL_ADAPTER_DIR)
        train_args = TrainingArguments(
            output_dir=os.path.join(OUTPUT_DIR, "training_pilot"),
            max_steps=TRAIN_PILOT_MAX_STEPS,
            per_device_train_batch_size=TRAIN_PILOT_BATCH_SIZE,
            gradient_accumulation_steps=TRAIN_PILOT_GRAD_ACCUM,
            learning_rate=TRAIN_PILOT_LEARNING_RATE,
            warmup_ratio=0.03,
            max_grad_norm=0.3,
            fp16=True,
            bf16=False,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            optim="paged_adamw_8bit",
            logging_steps=20,
            save_strategy="steps",
            save_steps=TRAIN_PILOT_SAVE_STEPS,
            save_total_limit=8,
            report_to="none",
            remove_unused_columns=False,
            dataloader_num_workers=0,
            seed=SEED,
            data_seed=SEED,
        )
        trainer = TaskWeightedTrainer(
            model=train_model,
            args=train_args,
            train_dataset=MultiHypothesisTrainingDataset(task_records),
            data_collator=MultiHypothesisTrainingCollator(processor),
        )
        trainer.train()
        final_dir = os.path.join(OUTPUT_DIR, "training_pilot", "final_adapter")
        train_model.save_pretrained(final_dir)
        processor.save_pretrained(final_dir)
        print("saved:", final_dir)
else:
    print("Training pilot is disabled. Enable RUN_TRAIN_RECORD_SELECTION first, then RUN_TRAINING_PILOT only if inference-only C improves on holdout.")


In [ ]:
# 11) Run config and artifact summary
run_config = {
    "experiment": "qwen2vl_multi_hypothesis_refine_v1",
    "run_id": RUN_ID,
    "initial_adapter": INITIAL_ADAPTER_DIR,
    "adapter_fingerprint": adapter_fingerprint(INITIAL_ADAPTER_DIR),
    "model_repo_id": MODEL_REPO_ID,
    "model_id": MODEL_ID,
    "split_dir": SPLIT_DIR,
    "split_hashes": {name: ids_hash(ids) for name, ids in split_ids.items()},
    "prompt_version": PROMPT_VERSION,
    "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS,
    "pairwise_bidirectional": PAIRWISE_BIDIRECTIONAL,
    "reference_config": reference_config_summary,
    "k_text_hypotheses": K_TEXT_HYPOTHESES,
    "weight_presets": WEIGHT_PRESETS,
    "uncertain_quantile_grid": UNCERTAIN_QUANTILE_GRID,
    "max_verifications_grid": MAX_VERIFICATIONS_GRID,
    "verify_margin_grid": VERIFY_MARGIN_GRID,
    "switch_margin_grid": SWITCH_MARGIN_GRID,
    "order_switch_gain_grid": ORDER_SWITCH_GAIN_GRID,
    "seed": SEED,
    "train_rows_for_records": TRAIN_ROWS_FOR_RECORDS,
    "train_pilot_max_steps": TRAIN_PILOT_MAX_STEPS,
    "train_pilot_save_steps": TRAIN_PILOT_SAVE_STEPS,
    "train_pilot_learning_rate": TRAIN_PILOT_LEARNING_RATE,
    "output_dir": OUTPUT_DIR,
    "cache_root": CACHE_ROOT,
}
save_json(run_config, os.path.join(RUN_ROOT, "run_config.json"))
save_json(run_config, os.path.join(OUTPUT_DIR, "run_config.json"))

best_config = {
    "experiment": "qwen2vl_multi_hypothesis_refine_v1",
    "selected_from": "tuning150",
    "best_tuning": best if "best" in globals() else None,
    "holdout": holdout_summary if "holdout_summary" in globals() else None,
    "run_config": run_config,
}
save_json(best_config, os.path.join(OUTPUT_DIR, "best_config.json"))
print(json.dumps(best_config, ensure_ascii=False, indent=2)[:4000])
print("Artifacts in:", OUTPUT_DIR)


## Notes

- The verifier receives only the disputed pair images and caption. It does not receive the current top-1 order, structured score, or previous decision.
- The conservative revision policy keeps baseline A unless joint evidence, verifier confidence, and score gain are all strong enough.
- Holdout is evaluated once with the best tuning150 configuration. Do not retune thresholds after reading holdout results.
- Training record selection is intentionally disabled by default and should only be enabled if inference-only C passes the holdout success criteria.
